# Continual Learning Routing — End-to-End Experiments

Compares five routing strategies on Split-CIFAR100. Each run is logged locally (Hydra writes to `outputs/<date>/<time>/` and result artifacts to `outputs/results/<name>/`) and to W&B (`wandb.mode=online`).

| # | Method | Notes |
|---|---|---|
| 1 | **Prototype + per-expert replay** | proposed method, default replay buffer |
| 2 | **Prototype + global replay** | replay-buffer ablation |
| 3 | **Top-1 fixed** | baseline |
| 4 | **Top-2 fixed** | baseline |
| 5 | **Top-3 fixed** | baseline |

**Runtime:** each run is 10 tasks × 5 epochs on a frozen ViT-small. Plan for ~30–60 min/run on a single GPU and several hours on CPU.

**W&B:** authenticate once with `wandb login` in your shell before running. To disable W&B for a quick smoke test, set `WANDB_MODE = "disabled"` in the setup cell.


## 0. Colab bootstrap (run this first)

If you're on Colab, this cell clones the repo, installs the package in editable mode, installs requirements, and (optionally) logs in to W&B. Locally it's a no-op.

**Before running on Colab:**

1. **Enable a GPU**: Runtime → Change runtime type → T4 GPU (or better).
2. **Set `REPO_URL`** below to your repo's clone URL. If your repo is private, use an HTTPS URL with a personal access token, or clone via SSH after configuring keys.
3. **W&B (optional)**: add a Colab Secret named `WANDB_API_KEY` (key icon in the left sidebar), or set `WANDB_MODE = "offline"` / `"disabled"` in the next cell to skip W&B entirely.


In [ ]:
import os
import subprocess
import sys
from pathlib import Path

# --- Edit these for your environment -------------------------------------
REPO_URL = "https://github.com/YOUR_USER/clr-routing.git"  # <-- change me
REPO_DIR = Path("/content/clr-routing")                     # clone target on Colab
# -------------------------------------------------------------------------

IN_COLAB = "google.colab" in sys.modules

if IN_COLAB:
    # 1. Clone the repo if it isn't already there.
    if not REPO_DIR.exists():
        print(f"Cloning {REPO_URL} -> {REPO_DIR}")
        subprocess.check_call(["git", "clone", REPO_URL, str(REPO_DIR)])
    else:
        print(f"Repo already at {REPO_DIR} (skipping clone)")

    # 2. cwd into the repo so the existing PROJECT_ROOT logic in the next cell
    #    resolves correctly (it looks for ./scripts/train.py).
    os.chdir(REPO_DIR)
    print("cwd:", Path.cwd())

    # 3. Install the package in editable mode if it has a setup file. This is
    #    what fixes the `ModuleNotFoundError: No module named 'clr_routing'`
    #    that subprocess.run hits — the spawned interpreter has no idea about
    #    the project layout otherwise. (We also set PYTHONPATH below as a
    #    belt-and-suspenders fallback.)
    if (REPO_DIR / "pyproject.toml").exists() or (REPO_DIR / "setup.py").exists():
        print("Installing package (pip install -e .)...")
        subprocess.check_call(
            [sys.executable, "-m", "pip", "install", "-q", "-e", "."]
        )
    else:
        print("No pyproject.toml/setup.py found — relying on PYTHONPATH only")

    # 4. Install requirements.txt if present.
    req = REPO_DIR / "requirements.txt"
    if req.exists():
        print("Installing requirements.txt...")
        subprocess.check_call(
            [sys.executable, "-m", "pip", "install", "-q", "-r", str(req)]
        )

    # 5. Pull a W&B API key from Colab Secrets if available.
    try:
        from google.colab import userdata  # type: ignore
        key = userdata.get("WANDB_API_KEY")
        if key:
            os.environ["WANDB_API_KEY"] = key
            print("WANDB_API_KEY loaded from Colab Secrets")
        else:
            print("No WANDB_API_KEY in Colab Secrets — set WANDB_MODE='offline' "
                  "in the next cell if you don't want to use W&B.")
    except Exception as e:
        print(f"Could not read Colab Secrets ({e}); set WANDB_API_KEY manually "
              "or use WANDB_MODE='offline'.")
else:
    print("Not on Colab — assuming the local environment is already set up.")


## 1. Setup


In [ ]:
from __future__ import annotations

import json
import os
import subprocess
import sys
from pathlib import Path

import matplotlib as mpl
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns

# Resolve project root whether this notebook is run from the repo root or notebooks/.
PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / "scripts" / "train.py").exists():
    PROJECT_ROOT = PROJECT_ROOT.parent
assert (PROJECT_ROOT / "scripts" / "train.py").exists(), (
    f"Could not locate scripts/train.py from {Path.cwd()}. "
    "Did the bootstrap cell above run successfully?"
)

RESULTS_ROOT = PROJECT_ROOT / "outputs" / "results"
FIG_DIR = PROJECT_ROOT / "outputs" / "figures"
FIG_DIR.mkdir(parents=True, exist_ok=True)

VENV_PYTHON = PROJECT_ROOT / ".venv" / "bin" / "python"
PYTHON = str(VENV_PYTHON) if VENV_PYTHON.exists() else sys.executable

# Set WANDB_MODE here so it overrides every subprocess uniformly.
WANDB_MODE = "online"  # 'online' | 'offline' | 'disabled'

# Environment for subprocesses. PYTHONPATH must include PROJECT_ROOT so the
# spawned `python scripts/train.py` can do `import clr_routing` — running a
# script under scripts/ puts scripts/ on sys.path, not the project root.
# (Redundant if `pip install -e .` succeeded in the bootstrap cell, but cheap
# insurance.)
SUBPROC_ENV = os.environ.copy()
SUBPROC_ENV["PYTHONPATH"] = (
    str(PROJECT_ROOT) + os.pathsep + SUBPROC_ENV.get("PYTHONPATH", "")
)

print("project root:", PROJECT_ROOT)
print("python      :", PYTHON)
print("results dir :", RESULTS_ROOT)
print("figures dir :", FIG_DIR)
print("wandb mode  :", WANDB_MODE)


## 2. Experiment definitions

Each entry below is one run. The `overrides` list is passed verbatim to Hydra; everything else falls back to `configs/base.yaml`. The display order, line color, and bar order in every plot below come from this list — keep it stable across re-runs.


In [ ]:
EXPERIMENTS = [
    {
        "name": "proto_per_expert",
        "label": "Prototype + per-expert replay (ours)",
        "overrides": [
            "experiment=proto_routing",
            "replay.global_replay=false",
            "wandb.name=proto_per_expert",
        ],
    },
    {
        "name": "proto_global",
        "label": "Prototype + global replay",
        "overrides": [
            "experiment=proto_routing",
            "replay.global_replay=true",
            "wandb.name=proto_global",
        ],
    },
    {
        "name": "baseline_top1",
        "label": "Fixed top-1",
        "overrides": ["experiment=baseline_top1", "wandb.name=baseline_top1"],
    },
    {
        "name": "baseline_top2",
        "label": "Fixed top-2",
        "overrides": ["experiment=baseline_top2", "wandb.name=baseline_top2"],
    },
    {
        "name": "baseline_top3",
        "label": "Fixed top-3",
        "overrides": ["experiment=baseline_top3", "wandb.name=baseline_top3"],
    },
]
EXPERIMENT_NAMES = [e["name"] for e in EXPERIMENTS]
EXPERIMENT_LABELS = {e["name"]: e["label"] for e in EXPERIMENTS}


## 3. Run all experiments

Reuses `scripts/train.py` (the same Hydra entry point you'd hit from the CLI), so local logs and W&B logs flow exactly the way they do for a normal training run. A run is skipped if `outputs/results/<name>/summary.json` already exists; flip `FORCE_RERUN = True` to retrain.


In [ ]:
FORCE_RERUN = False


def has_results(name: str) -> bool:
    return (RESULTS_ROOT / name / "summary.json").exists()


def run_experiment(spec: dict, force_rerun: bool = False) -> None:
    name = spec["name"]
    if has_results(name) and not force_rerun:
        print(f"[skip] {name} — summary.json already exists")
        return
    cmd = [
        PYTHON,
        "-u",  # unbuffered: prints show up live in the notebook
        "scripts/train.py",
        *spec["overrides"],
        f"wandb.mode={WANDB_MODE}",
    ]
    print(f"\n[run ] {name}")
    print(f"       {' '.join(cmd)}\n")

    # Stream stdout+stderr line-by-line so training progress is visible while
    # the run is happening — and so we actually see the traceback if it dies.
    proc = subprocess.Popen(
        cmd,
        cwd=str(PROJECT_ROOT),
        env=SUBPROC_ENV,
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,  # merge stderr into stdout for ordered output
        text=True,
        bufsize=1,
    )
    assert proc.stdout is not None
    tail: list[str] = []
    for line in proc.stdout:
        print(line, end="")
        tail.append(line)
        if len(tail) > 50:
            tail.pop(0)
    rc = proc.wait()
    if rc != 0:
        raise RuntimeError(
            f"{name} failed with exit code {rc}. Last {len(tail)} lines of output:\n"
            + "".join(tail)
        )


for spec in EXPERIMENTS:
    run_experiment(spec, force_rerun=FORCE_RERUN)

print("\nAll experiments finished.")


## 4. Load results

Each run writes:
- `summary.json` — headline metrics (`avg_accuracy`, `avg_forgetting`, `bwt`, `per_task_accuracy`)
- `accuracy_matrix.npy` — `(num_tasks, num_tasks)` matrix where `M[i, j]` is accuracy on task `j` after training task `i`
- `task_expert_similarity.npy` — task-prototype × expert-prototype cosine similarity (NaN for uninitialized rows/cols)


In [ ]:
def load_results() -> dict:
    runs = {}
    for spec in EXPERIMENTS:
        out = RESULTS_ROOT / spec["name"]
        with open(out / "summary.json") as f:
            summary = json.load(f)
        runs[spec["name"]] = {
            "label": spec["label"],
            "summary": summary,
            "matrix": np.load(out / "accuracy_matrix.npy"),
            "similarity": np.load(out / "task_expert_similarity.npy"),
        }
    return runs


runs = load_results()
print(f"loaded {len(runs)} runs: {list(runs)}")


## 5. Summary table


In [ ]:
rows = []
for name, run in runs.items():
    s = run["summary"]
    rows.append(
        {
            "Method": run["label"],
            "Avg accuracy ↑": s["avg_accuracy"],
            "Avg forgetting ↓": s["avg_forgetting"],
            "BWT ↑": s["bwt"],
        }
    )
summary_df = pd.DataFrame(rows).set_index("Method")
summary_df.style.format(
    {"Avg accuracy ↑": "{:.3f}", "Avg forgetting ↓": "{:.3f}", "BWT ↑": "{:.3f}"}
).background_gradient(cmap="Blues", subset=["Avg accuracy ↑"])


## 6. Plotting setup

A consistent palette and rcParams across every figure so they look uniform when dropped into a paper or slide deck. Figures are saved to `outputs/figures/` as both PDF (vector) and PNG (300 dpi).


In [ ]:
plt.style.use("seaborn-v0_8-whitegrid")
mpl.rcParams.update(
    {
        "figure.dpi": 110,
        "savefig.dpi": 300,
        "savefig.bbox": "tight",
        "font.family": "serif",
        "font.serif": ["DejaVu Serif", "Times New Roman", "STIX"],
        "font.size": 11,
        "axes.titlesize": 12,
        "axes.labelsize": 11,
        "axes.titleweight": "semibold",
        "xtick.labelsize": 10,
        "ytick.labelsize": 10,
        "legend.fontsize": 10,
        "legend.frameon": False,
        "axes.spines.top": False,
        "axes.spines.right": False,
        "axes.grid": True,
        "grid.alpha": 0.25,
        "grid.linestyle": "-",
        "lines.linewidth": 1.8,
        "lines.markersize": 5,
    }
)

PALETTE = sns.color_palette("colorblind", n_colors=len(EXPERIMENTS))
COLORS = {name: PALETTE[i] for i, name in enumerate(EXPERIMENT_NAMES)}


def save(fig: plt.Figure, stem: str) -> None:
    fig.savefig(FIG_DIR / f"{stem}.pdf")
    fig.savefig(FIG_DIR / f"{stem}.png")


## 7. Headline metrics

Side-by-side comparison of the three end-of-stream summary metrics. Arrows in the titles indicate the desirable direction.


In [ ]:
def headline_bar_chart() -> plt.Figure:
    metrics = [
        ("avg_accuracy", "Avg. accuracy  ↑"),
        ("avg_forgetting", "Avg. forgetting  ↓"),
        ("bwt", "Backward transfer  ↑"),
    ]
    fig, axes = plt.subplots(1, 3, figsize=(13.5, 3.8))
    for ax, (key, title) in zip(axes, metrics):
        names = list(runs)
        labels = [runs[n]["label"] for n in names]
        values = [runs[n]["summary"][key] for n in names]
        bars = ax.bar(
            range(len(names)),
            values,
            color=[COLORS[n] for n in names],
            edgecolor="black",
            linewidth=0.6,
        )
        for bar, v in zip(bars, values):
            ax.text(
                bar.get_x() + bar.get_width() / 2,
                v,
                f"{v:.3f}",
                ha="center",
                va="bottom" if v >= 0 else "top",
                fontsize=9,
            )
        ax.set_xticks(range(len(names)))
        ax.set_xticklabels(labels, rotation=25, ha="right")
        ax.set_title(title)
        ax.axhline(0, color="black", linewidth=0.6)
        ax.margins(y=0.18)
    fig.tight_layout()
    save(fig, "headline_metrics")
    return fig


_ = headline_bar_chart()


## 8. Average accuracy over the task stream

For each method, plots `mean_{j ≤ k} M[k, j]` against `k` — the standard “average accuracy on seen tasks” curve in continual learning papers.


In [ ]:
def avg_accuracy_curve() -> plt.Figure:
    fig, ax = plt.subplots(figsize=(7.2, 4.4))
    for name, run in runs.items():
        m = run["matrix"]
        n_tasks = m.shape[0]
        avg = np.array([m[k, : k + 1].mean() for k in range(n_tasks)])
        ax.plot(
            range(1, n_tasks + 1),
            avg,
            marker="o",
            label=run["label"],
            color=COLORS[name],
        )
    ax.set_xlabel("Tasks seen")
    ax.set_ylabel("Avg. accuracy on seen tasks")
    ax.set_title("Continual accuracy vs. task index")
    ax.set_xticks(range(1, runs[next(iter(runs))]["matrix"].shape[0] + 1))
    ax.legend(loc="best")
    fig.tight_layout()
    save(fig, "avg_accuracy_curve")
    return fig


_ = avg_accuracy_curve()


## 9. Per-method accuracy matrices

The full `(num_tasks, num_tasks)` accuracy matrix per method. Cells above the diagonal (tasks not yet seen) are masked. A common color scale lets you read forgetting and forward-transfer at a glance.


In [ ]:
def accuracy_matrix_grid() -> plt.Figure:
    n = len(runs)
    fig, axes = plt.subplots(1, n, figsize=(3.2 * n, 3.6), sharey=True)
    if n == 1:
        axes = [axes]
    im = None
    for ax, (name, run) in zip(axes, runs.items()):
        m = run["matrix"]
        mask = np.triu(np.ones_like(m, dtype=bool), k=1)
        masked = np.where(mask, np.nan, m)
        im = ax.imshow(masked, cmap="viridis", vmin=0, vmax=1, aspect="auto")
        ax.set_title(run["label"], fontsize=10)
        ax.set_xlabel("Eval task")
        ax.grid(False)
    axes[0].set_ylabel("After training task")
    cb = fig.colorbar(im, ax=axes, fraction=0.025, pad=0.02)
    cb.set_label("Accuracy")
    save(fig, "accuracy_matrices")
    return fig


_ = accuracy_matrix_grid()


## 10. Final per-task accuracy

How well each method retains accuracy on each individual task after training the full stream. The last row of the accuracy matrix.


In [ ]:
def final_per_task_accuracy() -> plt.Figure:
    fig, ax = plt.subplots(figsize=(7.2, 4.4))
    for name, run in runs.items():
        last = run["matrix"][-1]
        ax.plot(
            range(len(last)),
            last,
            marker="o",
            label=run["label"],
            color=COLORS[name],
        )
    ax.set_xlabel("Task index")
    ax.set_ylabel("Final accuracy")
    ax.set_title("Final accuracy per task (after training all tasks)")
    ax.set_xticks(range(len(last)))
    ax.legend(loc="best")
    fig.tight_layout()
    save(fig, "final_per_task_accuracy")
    return fig


_ = final_per_task_accuracy()


All figures are saved to `outputs/figures/` as both PDF (paper-ready) and PNG (slide-ready):

- `headline_metrics.{pdf,png}`
- `avg_accuracy_curve.{pdf,png}`
- `accuracy_matrices.{pdf,png}`
- `final_per_task_accuracy.{pdf,png}`
